# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

Open a copy in Colab or import it into Kaggle from the button above. Choose a runtime with the hardware you need; the link does not provision a TPU.

The setup cell does not install or replace PyTorch/TorchTPU. Before a native TPU run, the setup cell reports installed `torch-tpu` and `libtpu` versions and suggests a runtime with TorchTPU if either is missing. TrainLM checks package compatibility before loading data.

Open this notebook on a **single-device native TorchTPU runtime**. TrainLM prepares the tokenizer and data, trains a randomly initialized model, evaluates it, and saves the export and run report.

**Model architecture:** `model_type="gpt2"` selects Hugging Face GPT-2. TrainLM builds a `GPT2Config` from the values below and creates a `GPT2LMHeadModel` with random weights; it does not load a pretrained GPT-2 checkpoint. The tokenizer is learned from the training text, so TrainLM sets the model vocabulary and special-token IDs from that tokenizer.

The default 10,000-token run is only a workflow smoke test. It does not produce a useful model or certify TPU performance. Increase `max_tokens` for longer runs. For CPU/CUDA, change `accelerator` to `auto`.

Advanced settings and checkpoint recovery: [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch).

In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
    "tokenizers>=0.21,<1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
from importlib.metadata import PackageNotFoundError, version

for package_name in ("torch-tpu", "libtpu"):
    try:
        print(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}=MISSING; native TPU needs a runtime that provides TorchTPU")


In [ ]:
from trainlm import TrainLMPretrainingConfig, run_pretraining
RUN_DIR = "trainlm-pretraining"
MAX_TOKENS = 10_000
RESUME_FROM_CHECKPOINT = None  # Set to "latest" to continue a saved run.
EXTEND_TRAINING_BUDGET = False  # Set True only when increasing MAX_TOKENS.
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR,
    max_tokens=MAX_TOKENS,  # Increase this for a longer run.
    sequence_length=128,
    model_type="gpt2",  # Hugging Face GPT-2 architecture, random initialization.
    model_embedding_size=256,
    model_layers=4,
    model_attention_heads=4,
    accelerator="native_tpu",
)
result = run_pretraining(
    config,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    extend_training_budget=EXTEND_TRAINING_BUDGET,
    show_progress=True,
)
print(result.summary())
print(
    f"architecture: {result.report['model']['model_class']} "
    f"({result.report['model']['model_type']})"
)
print(f"resumed_from: {result.report['training']['resume_from_checkpoint']}")
print(f"checkpoint: {result.checkpoint_path}")
print(f"report: {result.report_path}")
print(f"model:  {result.export_path}")


## Optional: verify checkpoint resume

After a successful first run, keep the same runtime, `RUN_DIR`, model, data, and package versions. In the training cell, change only these values, then rerun that cell:

```python
MAX_TOKENS = 12_000
RESUME_FROM_CHECKPOINT = "latest"
EXTEND_TRAINING_BUDGET = True
```

This adds a small training segment from the newest saved checkpoint. A successful resume should report more steps/tokens and show the checkpoint in `resumed_from`. The default cosine schedule and automatic warmup are required for budget extension. If a runtime reset, keep the **full run directory** on persistent storage; the model artifact ZIP alone does not include training checkpoints or packed data.

For a new training run instead, leave `RESUME_FROM_CHECKPOINT = None` and `EXTEND_TRAINING_BUDGET = False`.

## Save the run artifacts

This creates a ZIP with the run report, model export (when available), and resume helper.

In [ ]:
from trainlm import bundle_pretraining_artifacts

bundle = bundle_pretraining_artifacts(result.report_path)
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))